# 4. Realized Semibeta (실현 반베타)

지금까지는 **한 종목**의 변동을 부호별로 쪼갰어. 이번엔 **종목과 시장, 두 개의 움직임**을 부호별로 쪼개는 거야. 순서는 베타의 기초 → realized beta → 부호 분해 유도 → 숫자 예시 → 실증 → 함정 → 코드야.

---

## 0단계: 베타가 뭔지부터

CAPM의 베타는 "시장이 1% 움직일 때 이 종목이 평균적으로 몇 % 움직이나"야.

$$
\beta_i = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}
$$

- 분자 **공분산**: 종목 수익률 $r_i$와 시장 수익률 $r_m$이 **같이 움직이는 정도**야.
- 분모 **시장 분산**: 시장 자체의 움직임 크기로 나눠서 "시장 1단위당" 반응으로 바꿔줘.

공분산은 모멘트 관점에서 보면 **두 변수를 섞은 2차 모멘트**야.

$$
\text{Cov}(r_i, r_m) = E\big[(r_i - \mu_i)(r_m - \mu_m)\big]
$$

분산이 $E[(X-\mu)^2] = E[(X-\mu)(X-\mu)]$, 즉 **자기 자신과 곱한 것**이라면, 공분산은 **다른 변수와 곱한 것**이야.

---

## 1단계: Realized Beta — 장중 데이터로 베타 계산

RV를 만들 때와 똑같은 논리를 적용해. 10분봉 평균은 사실상 0이니 평균을 빼지 않고, 같은 시간대 봉끼리 곱해서 더해.

$$
RCov_t = \sum_{k=1}^{N} r_{i,k}\, r_{m,k}, \qquad RV_{m,t} = \sum_{k=1}^{N} r_{m,k}^2
$$

$$
\beta_t = \frac{\sum_{k} r_{i,k}\, r_{m,k}}{\sum_{k} r_{m,k}^2}
$$

여기서 $k$는 봉 번호(09:10, 09:20, …)야. **같은 시각의 종목 봉과 시장 봉을 짝지어 곱한다**는 게 핵심이야.

---

## 2단계: 문제 — 베타는 위아래를 구분하지 않아

베타 1.5인 종목이 두 개 있다고 하자.

- 종목 A: 시장이 **떨어질 때** 같이 크게 떨어지고, 오를 때는 조금만 따라 올라.
- 종목 B: 시장이 **오를 때** 같이 크게 오르고, 떨어질 때는 조금만 따라 내려.

투자자 입장에서 A는 무섭고 B는 매력적이야. 폭락장에서 손실을 키우는 건 A니까. 그런데 **베타는 둘 다 1.5로 똑같아.** 공분산이 위아래 움직임을 한 숫자로 섞어버리기 때문이야. 앞의 RV가 급등일과 급락일을 구분 못 했던 것과 **정확히 같은 문제**지.

---

## 3단계: 해결책 — 각 수익률을 양수 부분과 음수 부분으로 쪼개기

semivariance에서 했던 걸 두 변수 각각에 적용해. 어떤 수익률 r이든 이렇게 쪼갤 수 있어.

$$
r^{+} = \max(r, 0), \qquad r^{-} = \min(r, 0)
$$

예를 들어:

- $r = +0.3$ → $r^+ = 0.3$, $r^- = 0$
- $r = -0.5$ → $r^+ = 0$, $r^- = -0.5$

어느 경우든 **둘 중 하나는 반드시 0**이고, 둘을 더하면 원래 값이 돼.

$$
r = r^{+} + r^{-}
$$

---

## 4단계: 곱을 전개하면 네 조각이 나온다 (핵심 유도)

realized beta의 분자에 있는 곱 $r_{i,k}\, r_{m,k}$에 3단계를 대입하자. 표기를 줄이려고 봉 번호 k는 생략할게.

$$
r_i\, r_m = (r_i^{+} + r_i^{-})(r_m^{+} + r_m^{-})
$$

괄호를 전개하면 (중학교 때 배운 $(a+b)(c+d) = ac+ad+bc+bd$ 그대로야):

$$
r_i\, r_m = \underbrace{r_i^{+} r_m^{+}}_{\text{①}} + \underbrace{r_i^{-} r_m^{-}}_{\text{②}} + \underbrace{r_i^{+} r_m^{-}}_{\text{③}} + \underbrace{r_i^{-} r_m^{+}}_{\text{④}}
$$

각 항이 언제 0이 아닌지 보면:

| 항 | 종목 | 시장 | 부호 | 의미 |
|---|---|---|---|---|
| ① $r_i^+ r_m^+$ | ↑ | ↑ | + | 같이 오름 |
| ② $r_i^- r_m^-$ | ↓ | ↓ | + (음×음) | **같이 떨어짐** |
| ③ $r_i^+ r_m^-$ | ↑ | ↓ | − | 시장 떨어질 때 종목 오름 |
| ④ $r_i^- r_m^+$ | ↓ | ↑ | − | 시장 오를 때 종목 떨어짐 |

한 봉 안에서 종목과 시장의 부호 조합은 넷 중 하나뿐이니까, **매 봉마다 네 항 중 정확히 하나만 살아남아.** 결국 모든 봉을 **2×2 사분면**으로 분류하는 거야.

---

## 5단계: 네 개의 semibeta 정의

네 항을 각각 합하고 시장 RV로 나누면 끝이야. ③과 ④는 항상 음수라서, **해석하기 쉽게 마이너스를 붙여 양수로** 만들어.

$$
\beta^{N}_t = \frac{\sum_k r_{i,k}^{-}\, r_{m,k}^{-}}{RV_{m,t}} \quad \text{(같이 하락)}
$$

$$
\beta^{P}_t = \frac{\sum_k r_{i,k}^{+}\, r_{m,k}^{+}}{RV_{m,t}} \quad \text{(같이 상승)}
$$

$$
\beta^{M+}_t = -\frac{\sum_k r_{i,k}^{+}\, r_{m,k}^{-}}{RV_{m,t}} \quad \text{(시장↓, 종목↑)}
$$

$$
\beta^{M-}_t = -\frac{\sum_k r_{i,k}^{-}\, r_{m,k}^{+}}{RV_{m,t}} \quad \text{(시장↑, 종목↓)}
$$

N은 Negative(둘 다 음), P는 Positive(둘 다 양), M은 Mixed(부호 엇갈림)의 약자야. 첨자 +/−는 종목 쪽 부호를 가리켜. 이 표기는 Bollerslev, Patton, Quaedvlieg(2022)를 따른 거지만, 논문을 직접 읽을 때는 첨자 정의를 원문에서 한 번 확인해. 헷갈리기 쉬운 부분이라 사분면 설명으로 기억하는 게 안전해.

네 개를 합치면 원래 베타로 **정확히** 돌아와.

$$
\boxed{\beta_t = \beta^{N}_t + \beta^{P}_t - \beta^{M+}_t - \beta^{M-}_t}
$$

마이너스를 붙여서 정의했으니 다시 빼주는 거야. semivariance가 $RV = RS^+ + RS^-$로 RV를 분해했듯이, semibeta는 **베타를 네 조각으로 분해**한 거야.

---

## 6단계: 숫자 예시

N = 5, 시장 수익률(%)은 다음과 같아.

$$
r_m = [0.2,\ -0.3,\ 0.1,\ -0.4,\ 0.2], \qquad RV_m = 0.04+0.09+0.01+0.16+0.04 = 0.34
$$

**종목 A (하락장에 크게 같이 빠짐):** $r_A = [0.1,\ -0.6,\ 0.1,\ -0.8,\ 0.1]$

| 봉 | $r_A$ | $r_m$ | 사분면 | 곱 |
|---|---|---|---|---|
| 1 | 0.1 | 0.2 | P | 0.02 |
| 2 | −0.6 | −0.3 | N | 0.18 |
| 3 | 0.1 | 0.1 | P | 0.01 |
| 4 | −0.8 | −0.4 | N | 0.32 |
| 5 | 0.1 | 0.2 | P | 0.02 |

- $\beta^N = 0.50 / 0.34 = 1.47$
- $\beta^P = 0.05 / 0.34 = 0.15$
- 혼합 항은 없음
- $\beta = 1.62$

**종목 B (상승장에 크게 같이 오름):** $r_B = [1.0,\ -0.1,\ 1.0,\ -0.05,\ 1.0]$

- $\beta^P = (0.2 + 0.1 + 0.2)/0.34 = 1.47$
- $\beta^N = (0.03 + 0.02)/0.34 = 0.15$
- $\beta = 1.62$

| 종목 | β | β^N (같이 하락) | β^P (같이 상승) |
|---|---|---|---|
| A | **1.62** | **1.47** | 0.15 |
| B | **1.62** | 0.15 | **1.47** |

**베타는 완전히 같은데, 위험의 성격은 정반대야.** 2단계에서 말한 문제가 semibeta로 정확히 드러나지.

**종목 C (헤지형, 시장이 빠질 때 오름):** $r_C = [0.2,\ 0.3,\ 0.1,\ 0.4,\ 0.2]$

- 봉 1, 3, 5는 P: $(0.04 + 0.01 + 0.04)/0.34 = 0.26$
- 봉 2, 4는 시장↓·종목↑: $\beta^{M+} = -(0.3 \times -0.3 + 0.4 \times -0.4)/0.34 = 0.25/0.34 = 0.74$
- $\beta = 0.26 - 0.74 = -0.47$

음의 베타가 **"시장 하락 시 종목 상승"**이라는 혼합 사분면에서 나왔다는 걸 분해해서 확인할 수 있어.

---

## 7단계: 실증 결과 — 어느 조각에 가격이 매겨지나

Bollerslev, Patton, Quaedvlieg(2022, JFE)의 결론을 요약하면 이래.

- **시장이 하락할 때의 두 semibeta만 미래 수익률을 예측**했어.
  - 같이 하락($\beta^N$)이 클수록 → 미래 수익률이 **높아** (위험 프리미엄)
  - 시장↓·종목↑($\beta^{M+}$)이 클수록 → 미래 수익률이 **낮아** (헤지 가치)
- **시장이 상승할 때의 두 semibeta는 가격에 거의 반영되지 않았어.**

**경제적 해석:** 투자자가 진짜 두려워하는 건 "시장 전체가 무너질 때 내 종목도 같이 무너지는 것"이야. 그런 종목은 위험하니 보유하는 대가로 높은 기대수익을 요구해. 반대로 시장이 무너질 때 오르는 종목은 **보험** 역할을 하니, 투자자들이 비싸게 사주고 기대수익이 낮아져. 시장이 오를 때의 동조는 아무도 신경 쓰지 않아.

**이게 왜 중요한가:** 전통적인 CAPM 베타는 실증적으로 미래 수익률을 잘 설명하지 못하는 것으로 유명해(고베타 종목이 이론만큼 수익을 내지 못함). semibeta 관점에서 보면 이유가 설명돼. 베타는 **가격이 매겨지는 조각(하락장 동조)과 매겨지지 않는 조각(상승장 동조)을 섞어놓은 것**이라, 신호가 희석되는 거야.

---

## 8단계: 네 프로젝트에 쓸 때의 함정

**동시성 문제 (Epps 효과).** 두 변수의 곱을 쓰니까 **같은 시각의 가격**이어야 해. 소형주는 10분 동안 거래가 한 번도 없을 수 있고, 그러면 그 봉의 수익률이 0이었다가 다음 봉에 몰려서 반영돼. 시장 지수는 즉시 반영되는데 종목은 늦게 반영되니 곱이 엇갈려서, **공분산과 베타가 0 쪽으로 과소추정**돼. 이걸 Epps 효과라고 해. 10분봉이면 1분봉보다는 훨씬 덜하지만, 비유동 종목은 여전히 걸러야 해.

**일별 값은 너무 불안정해.** 하루 39개 봉을 네 사분면으로 나누면 한 사분면에 봉이 몇 개 안 들어가. 논문들도 보통 **한 달치 장중 데이터를 모아서** 월별 semibeta를 계산해. 분자와 분모를 각각 한 달 동안 합산한 뒤 나누는 게 맞아(RSJ 주간 집계 때와 같은 원리).

**시장 수익률을 무엇으로 쓸까.** KOSPI, KOSPI200, KOSDAQ 중 무엇을 기준으로 할지가 결과에 영향을 줘. 코스닥 종목에 KOSPI200을 시장으로 쓰면 베타가 체계적으로 낮게 나와. 최소한 시장 구분별로 맞는 지수를 쓰거나 이 선택을 명시해야 해.

**첫 봉과 오버나이트.** 여기서도 똑같아. 시가 동시호가 봉과 오버나이트 수익률은 따로 처리해.

---

## 계산 코드

```python
import numpy as np
import pandas as pd

def semibetas(ri: np.ndarray, rm: np.ndarray) -> pd.Series:
    """ri, rm: 같은 시각으로 정렬된 종목/시장 10분봉 수익률 (한 달치를 이어붙여도 됨)"""
    mask = ~(np.isnan(ri) | np.isnan(rm))
    ri, rm = ri[mask], rm[mask]
    ri_p, ri_n = np.maximum(ri, 0), np.minimum(ri, 0)
    rm_p, rm_n = np.maximum(rm, 0), np.minimum(rm, 0)
    rv_m = np.sum(rm ** 2)
    if rv_m == 0:
        return pd.Series(dtype=float)
    bN  =  np.sum(ri_n * rm_n) / rv_m   # 같이 하락
    bP  =  np.sum(ri_p * rm_p) / rv_m   # 같이 상승
    bMp = -np.sum(ri_p * rm_n) / rv_m   # 시장↓ 종목↑
    bMn = -np.sum(ri_n * rm_p) / rv_m   # 시장↑ 종목↓
    return pd.Series({'beta': bN + bP - bMp - bMn,
                      'beta_N': bN, 'beta_P': bP,
                      'beta_Mplus': bMp, 'beta_Mminus': bMn})

# df: [ticker, datetime, ret10m], mkt: [datetime, mret10m]
merged = df.merge(mkt, on='datetime', how='inner')   # 같은 시각끼리만 짝짓기
merged['month'] = merged['datetime'].dt.to_period('M')
monthly = (merged.groupby(['ticker', 'month'])
                 .apply(lambda g: semibetas(g['ret10m'].values, g['mret10m'].values))
                 .unstack())
```

검증 팁 하나. 계산 후 `beta`가 일반 realized beta($\sum r_i r_m / \sum r_m^2$)와 **정확히 일치하는지** 확인해봐. 5단계 박스 식이 항등식이니까 다르면 코드에 버그가 있는 거야.

---

## 지금까지 배운 것과의 연결

| 개념 | 무엇을 쪼개나 | 조각 수 |
|---|---|---|
| Realized semivariance | 한 종목의 **분산** | 2개 (상승/하락) |
| Signed jump variation | 위 두 조각의 **차이** | 1개 (방향) |
| Realized semibeta | 종목과 시장의 **공분산** | 4개 (사분면) |

관통하는 아이디어는 하나야. **제곱이나 곱을 하는 순간 사라지는 부호 정보를, 곱하기 전에 부호별로 쪼개서 살려두자.**

## 네 프로젝트와의 연결

semibeta는 **레짐 연구와 궁합이 특히 좋아.** 위기 국면에서는 종목들이 하락장에 한꺼번에 같이 빠지는 경향이 강해지는데(하락장 상관관계가 상승장보다 높다는 건 오래된 실증 결과야), 이건 곧 **$\beta^N$이 레짐에 따라 달라진다**는 뜻이야. 그러면 이런 질문을 던질 수 있어.

- 고변동 레짐에서 $\beta^N$의 위험 프리미엄이 커지는가?
- 한국 시장의 공매도 금지 기간에 $\beta^N$과 $\beta^{M+}$의 가격 반영이 달라졌는가?

개별 종목 수익률의 방향을 맞히는 것보다 훨씬 이론적 근거가 탄탄한 질문이야.

---

**한 줄 요약:** Realized semibeta는 종목과 시장의 10분봉 수익률을 각각 양수·음수 부분으로 쪼갠 뒤 곱을 전개해서, 베타를 **같이 하락 / 같이 상승 / 엇갈림 두 가지**의 네 조각으로 분해한 거야. 실증적으로는 **시장 하락 시의 조각만** 미래 수익률에 가격이 매겨져.

다음 5번 **고유변동성 퍼즐**은 베타(시장과 같이 움직이는 부분)로 설명되지 **않는** 나머지 변동성 이야기야. 방금 배운 베타 개념이 바로 이어지니 좋은 타이밍이야. 준비되면 말해줘.